# 🔢 Notebook 05 — Embeddings & FAISS Vector Store
**Healthcare RAG-Powered Medical Q&A Assistant**
**eyouth × DEPI | Microsoft Machine Learning Track | 2026**

---

### 🎯 Objectives
- Load the labelled dataset
- Build formatted text chunks (question + context + answer)
- Generate dense embeddings with `S-PubMedBert-MS-MARCO (biomedical domain)`
- Build a FAISS `IndexFlatIP` index
- Persist index + chunk mapping to disk
- Sanity-check retrieval with 5 test queries (top-3)
- Measure retrieval latency (KPI: < 500ms)

### 📋 Deliverables
- `data/embeddings/faiss_index/pubmedqa_index_flatip.faiss`
- `data/embeddings/faiss_index/chunk_mapping.csv`
- `data/embeddings/faiss_index/chunk_mapping.pkl`

---

## 1. Imports & Setup

In [1]:
import os
import pickle
import time
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

print("✅ Imports successful")

✅ Imports successful


## 2. Load Data & Build Text Chunks

Each chunk combines **question + context + answer** into a single
retrieval unit. Including the context (medical abstract) is critical
because it contains the evidence a RAG pipeline needs to ground its answers.

In [2]:
data_path = "../data/processed/pubmedqa_labelled.csv"

df = pd.read_csv(data_path)
print(f"✅ Loaded: {data_path}")
print(f"   Total rows: {len(df):,}")
print(f"   Columns: {list(df.columns)}")

# Validate required columns
required_columns = {"question", "context", "answer"}
missing = required_columns - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Drop rows with missing text
before = len(df)
df = df.dropna(subset=["question", "context", "answer"]).copy()
df["question"] = df["question"].astype(str).str.strip()
df["context"] = df["context"].astype(str).str.strip()
df["answer"] = df["answer"].astype(str).str.strip()
df = df[(df["question"] != "") & (df["context"] != "") & (df["answer"] != "")]
df = df.reset_index(drop=True)
print(f"   Rows after cleaning: {len(df):,} (dropped {before - len(df)})")
# -- Deduplicate by question ---------------------------------------------------
before_dedup = len(df)
df = df.drop_duplicates(subset=["question"], keep="first").copy().reset_index(drop=True)
dupes_removed = before_dedup - len(df)
print(f"   Duplicates removed: {dupes_removed}")
print(f"   Rows after dedup: {len(df):,}")
print("   Category distribution:")
for cat, count in df["category"].value_counts().items():
    print(f"     {cat}: {count:,} ({count/len(df)*100:.1f}%)")


✅ Loaded: ../data/processed/pubmedqa_labelled.csv
   Total rows: 211,186
   Columns: ['question', 'context', 'answer', 'category']
   Rows after cleaning: 211,186 (dropped 0)
   Duplicates removed: 78
   Rows after dedup: 211,108
   Category distribution:
     Medication: 71,522 (33.9%)
     Treatment: 48,550 (23.0%)
     Diagnosis: 31,911 (15.1%)
     General: 28,243 (13.4%)
     Prevention: 22,166 (10.5%)
     Symptoms: 8,716 (4.1%)


In [3]:
from sklearn.model_selection import train_test_split

# -- Stratified Train/Holdout Split -------------------------------------------
EVAL_HOLDOUT_SIZE = 2000
RANDOM_SEED = 42

df_train, df_eval = train_test_split(
    df,
    test_size=EVAL_HOLDOUT_SIZE,
    stratify=df["category"],
    random_state=RANDOM_SEED,
)
df_train = df_train.reset_index(drop=True)
df_eval = df_eval.reset_index(drop=True)

print("Dataset split:")
print(f"   Training set:  {len(df_train):,} rows")
print(f"   Holdout set:   {len(df_eval):,} rows")
print()

print("   Holdout category distribution:")
for cat, count in df_eval["category"].value_counts().items():
    print(f"     {cat}: {count:,} ({count/len(df_eval)*100:.1f}%)")

# Save holdout for downstream evaluation
holdout_path = Path("../data/processed/eval_holdout.csv")
df_eval.to_csv(holdout_path, index=False)
print(f"\n   Holdout saved: {holdout_path}")


Dataset split:
   Training set:  209,108 rows
   Holdout set:   2,000 rows

   Holdout category distribution:
     Medication: 678 (33.9%)
     Treatment: 460 (23.0%)
     Diagnosis: 302 (15.1%)
     General: 267 (13.4%)
     Prevention: 210 (10.5%)
     Symptoms: 83 (4.2%)

   Holdout saved: ..\data\processed\eval_holdout.csv


## 3. Generate Embeddings

Using `sentence-transformers/S-PubMedBert-MS-MARCO (biomedical domain)`:
- 768-dimensional embeddings
- Fast inference, good semantic quality
- FAISS requires `float32` arrays

In [ ]:
# Build text chunks with RecursiveCharacterTextSplitter (chunk_size=700, overlap=150)
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""],
)

all_chunks = []
all_meta = []

for _, row in df_train.iterrows():
    full_text = (
        f"Question: {row['question']}\n"
        f"Context: {row['context']}\n"
        f"Answer: {row['answer']}"
    )
    sub_chunks = splitter.split_text(full_text)
    for chunk in sub_chunks:
        all_chunks.append(chunk)
        all_meta.append({
            "question": row["question"],
            "context": row["context"],
            "answer": row["answer"],
            "category": row.get("category", "Unknown"),
            "text_chunk": chunk,
        })

df_chunks = pd.DataFrame(all_meta)
text_chunks = all_chunks
n_chunks = len(text_chunks)
print(f"Built {n_chunks:,} text chunks (from {len(df_train):,} records)")
print(f"   Sample:\n{text_chunks[0][:300]}")

In [5]:
model_name = "pritamdeka/S-PubMedBert-MS-MARCO"
model = SentenceTransformer(model_name)
print(f"✅ Loaded model: {model_name}")
print(f"   Embedding dimension: {model.get_embedding_dimension()}")

✅ Loaded model: pritamdeka/S-PubMedBert-MS-MARCO
   Embedding dimension: 768


In [6]:
from tqdm import tqdm

batch_size = 64
embeddings_list = []

# Calculate total number of batches
n_batches = (n_chunks + batch_size - 1) // batch_size

print(f"⏳ Encoding {n_chunks:,} chunks in {n_batches} batches (batch_size={batch_size})...")
start_time = time.time()

# Process in batches to show detailed counter
for i in tqdm(range(0, n_chunks, batch_size), desc="Encoding chunks", unit="batch"):
    batch_texts = text_chunks[i:i+batch_size]
    batch_embeddings = model.encode(
        batch_texts,
        batch_size=batch_size,
        show_progress_bar=False,      # disable inner bar, we use outer tqdm
        convert_to_numpy=True,
        normalize_embeddings=True,     # normalize for cosine similarity
    )
    embeddings_list.append(batch_embeddings)

    # Optional: print intermediate count every 5000 rows
    processed = min(i + batch_size, n_chunks)
    if processed % 5000 == 0 or processed == n_chunks:
        print(f"   ✅ Processed {processed:,} / {n_chunks:,} rows")

# Concatenate all batches
embeddings = np.vstack(embeddings_list).astype(np.float32)


encoding_time = time.time() - start_time

print(f"\n✅ Encoding complete!")
print(f"   Shape: {embeddings.shape}")
print(f"   Dtype: {embeddings.dtype}")
print(f"   Time: {encoding_time:.1f}s ({encoding_time/n_chunks*1000:.1f}ms per chunk)")

⏳ Encoding 209,108 chunks in 3268 batches (batch_size=64)...


Encoding chunks:  19%|█▉        | 625/3268 [07:35<32:12,  1.37batch/s]

   ✅ Processed 40,000 / 209,108 rows


Encoding chunks:  38%|███▊      | 1250/3268 [15:13<24:35,  1.37batch/s]

   ✅ Processed 80,000 / 209,108 rows


...

Encoding chunks:  96%|█████████▌| 3125/3268 [38:06<01:45,  1.35batch/s]

   ✅ Processed 200,000 / 209,108 rows


Encoding chunks: 100%|██████████| 3268/3268 [39:49<00:00,  1.37batch/s]

   ✅ Processed 209,108 / 209,108 rows



✅ Encoding complete!
   Shape: (209108, 768)
   Dtype: float32
   Time: 2389.5s (11.4ms per chunk)


## 4. Build FAISS Index

`IndexFlatIP` — exact nearest-neighbor search using inner product (cosine similarity).
- No training needed
- Dimension inferred from embeddings
- Higher score = higher similarity (cosine)

In [7]:
d = embeddings.shape[1]

index = faiss.IndexFlatIP(d)
index.add(embeddings)

print(f"✅ FAISS index built")
print(f"   Dimension: {d}")
print(f"   Total vectors: {index.ntotal:,}")

✅ FAISS index built
   Dimension: 768
   Total vectors: 209,108


## 5. Save Index & Chunk Mapping

In [8]:
output_dir = Path("../data/embeddings/faiss_index")
output_dir.mkdir(parents=True, exist_ok=True)

index_path = output_dir / "pubmedqa_index_flatip.faiss"
mapping_csv_path = output_dir / "chunk_mapping.csv"
mapping_pkl_path = output_dir / "chunk_mapping.pkl"

# Save FAISS index
faiss.write_index(index, str(index_path))

# Save mapping table (chunk_id → question, context, answer, category, text_chunk)
# IMPORTANT: chunk_id values stay 0..(n_chunks-1) — they are positions in this FAISS index,
# NOT row numbers from the original labelled CSV. The eval set is held out separately.
mapping_df = df_chunks.copy()
mapping_df.insert(0, "chunk_id", np.arange(len(mapping_df), dtype=np.int32))

mapping_df.to_csv(mapping_csv_path, index=False)

with open(mapping_pkl_path, "wb") as f:
    pickle.dump(mapping_df, f)

print(f"✅ Saved FAISS index to: {index_path}")
print(f"   Vectors:   {index.ntotal:,}")
print(f"   File size: {os.path.getsize(index_path) / 1024**2:.1f} MB")
print(f"✅ Saved chunk mapping CSV to: {mapping_csv_path}")
print(f"✅ Saved chunk mapping PKL to: {mapping_pkl_path}")

# Final sanity: chunk count == FAISS vectors == df_train rows
assert index.ntotal == len(mapping_df) == len(df_chunks), \
    f"Mismatch: index={index.ntotal}, mapping={len(mapping_df)}, df_train={len(df_chunks)}"
print(f"✅ Sanity check passed: {index.ntotal:,} vectors == {len(mapping_df):,} mapping rows")

✅ Saved FAISS index to: ..\data\embeddings\faiss_index\pubmedqa_index_flatip.faiss
   Vectors:   209,108
   File size: 612.6 MB
✅ Saved chunk mapping CSV to: ..\data\embeddings\faiss_index\chunk_mapping.csv
✅ Saved chunk mapping PKL to: ..\data\embeddings\faiss_index\chunk_mapping.pkl
✅ Sanity check passed: 209,108 vectors == 209,108 mapping rows


In [3]:
# ── Auto-upload to HuggingFace ────────────────────────────────────────────
import os
import sys
sys.path.append(os.path.abspath('..'))

from dotenv import load_dotenv
load_dotenv()

from src.data.hub import upload_file

print("\n📤 Syncing to HuggingFace...")

# FAISS index + chunk mapping (rebuilt from stratified training set)
upload_file(

    "data/embeddings/faiss_index/pubmedqa_index_flatip.faiss",
    "embeddings/pubmedqa_index_flatip.faiss"
)
upload_file(
    "data/embeddings/faiss_index/chunk_mapping.pkl",
    "embeddings/chunk_mapping.pkl"
)
upload_file(
    "data/embeddings/faiss_index/chunk_mapping.csv",
    "embeddings/chunk_mapping.csv"
)

# Held-out eval set (2,000 rows, stratified, NOT in FAISS index)
upload_file(
    "data/processed/eval_holdout.csv",
    "processed/eval_holdout.csv"
)


📤 Syncing to HuggingFace...
  📤 Uploading: data/embeddings/faiss_index/pubmedqa_index_flatip.faiss (612.6 MB) → AbdoMatrix/healthcare-rag-data/embeddings/pubmedqa_index_flatip.faiss


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


  ✅ Uploaded: embeddings/pubmedqa_index_flatip.faiss
  📤 Uploading: data/embeddings/faiss_index/chunk_mapping.pkl (700.7 MB) → AbdoMatrix/healthcare-rag-data/embeddings/chunk_mapping.pkl


...

No files have been modified since last commit. Skipping to prevent empty commit.


  ✅ Uploaded: embeddings/chunk_mapping.pkl
  📤 Uploading: data/embeddings/faiss_index/chunk_mapping.csv (700.4 MB) → AbdoMatrix/healthcare-rag-data/embeddings/chunk_mapping.csv


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

  ✅ Uploaded: embeddings/chunk_mapping.csv
  📤 Uploading: data/processed/eval_holdout.csv (3.3 MB) → AbdoMatrix/healthcare-rag-data/processed/eval_holdout.csv
  ✅ Uploaded: processed/eval_holdout.csv


## 6. Sanity-Check Retrieval (Top-3) + Latency Measurement

**M2 KPI: FAISS returns results in < 500ms**

In [ ]:
test_queries = [
    "What are effective treatments for irritable bowel syndrome symptoms?",
    "Can hypothyroidism increase risk of fatty liver disease?",
    "Is laparoscopic prostatectomy superior to retropubic surgery?",
    "How accurate is diagnosis of acute otitis media in primary care?",
    "Does secondary isoniazid therapy reduce recurrent tuberculosis in HIV patients?",
]

# Encode queries and normalize them (to match indexed embeddings)
query_embeddings = model.encode(
    test_queries,
    batch_size=16,
    show_progress_bar=False,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
query_embeddings = np.asarray(query_embeddings, dtype=np.float32)

k = 3  # top-3 results

# ── Run retrieval with timing ────────────────────────────────────────────
latencies = []

for qi, query in enumerate(test_queries):
    # Time only the FAISS search (not encoding)
    start = time.perf_counter()
    D, I = index.search(query_embeddings[qi:qi+1], k)
    elapsed_ms = (time.perf_counter() - start) * 1000
    latencies.append(elapsed_ms)

    print("=" * 110)
    print(f"Query {qi + 1}: {query}")
    print(f"⏱️  Retrieval latency: {elapsed_ms:.2f}ms")
    print()

    for rank in range(k):
        idx = int(I[0, rank])
        dist = float(D[0, rank])
        chunk = mapping_df.loc[idx, "text_chunk"]

        print(f"  Top {rank + 1} | Chunk ID: {idx} | Score (IP): {dist:.4f}")
        print(f"  {chunk[:300]}")
        if len(chunk) > 300:
            print("  ... [truncated]")
        print("-" * 110)

# ── Latency summary ─────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("⏱️  LATENCY SUMMARY")
print("=" * 60)
print(f"  Min:    {min(latencies):.2f}ms")
print(f"  Max:    {max(latencies):.2f}ms")
print(f"  Mean:   {np.mean(latencies):.2f}ms")
print(f"  Median: {np.median(latencies):.2f}ms")
print(f"\n  Index size: {index.ntotal:,} vectors (held-out holdout: {EVAL_HOLDOUT_SIZE} rows)")
print()

if max(latencies) < 500:
    print("✅ KPI MET: All queries returned in < 500ms")
else:
    print("⚠️  KPI NOT MET: Some queries exceeded 500ms")

## ✅ Summary

| Item | Result |
|---|---|
| Text chunks | Question + Context + Answer |
| Embedding model | `S-PubMedBert-MS-MARCO (biomedical domain)` (768d) |
| FAISS index type | `IndexFlatIP` |
| Vectors stored | 209,108 |
| Sanity check | 5 queries × top-3 results |
| Latency KPI | < 500ms per query |

### Files Saved
- `data/embeddings/faiss_index/pubmedqa_index_flatip.faiss`
- `data/embeddings/faiss_index/chunk_mapping.csv`
- `data/embeddings/faiss_index/chunk_mapping.pkl`

---

### ➡️ Next Step
Open **`06_rag_pipeline.ipynb`** to build the LangChain RAG pipeline.